In [1]:
import pandas as pd

## Find Irainian Candidates

In [2]:
df_basics = pd.read_csv('../data/raw/title.basics.tsv.gz', sep='\t', low_memory=False)

In [4]:
df_akas = pd.read_csv('../data/raw/title.akas.tsv.gz', sep='\t', low_memory=False)

In [18]:
iran_candidates = df_akas[(df_akas['region'] == 'IR') & (df_akas['language'] == 'fa')]['titleId'].unique()

In [19]:
print(len(iran_candidates))

4503


In [20]:
region_counts = df_akas.groupby('titleId')['region'].nunique()
low_region_count = region_counts[region_counts <= 5].index

In [21]:
iran_titles_v3 = set(iran_candidates) & set(low_region_count) #intersection
print(len(iran_titles_v3))

1854


In [35]:
print(iran_titles_v3)

{'tt8114174', 'tt30128241', 'tt8347878', 'tt27757477', 'tt8348052', 'tt9591766', 'tt7510836', 'tt30990943', 'tt12063924', 'tt16764112', 'tt2970322', 'tt0382275', 'tt1847648', 'tt7903340', 'tt8988990', 'tt0292950', 'tt9031772', 'tt0400329', 'tt7247282', 'tt8394334', 'tt0253922', 'tt15258940', 'tt9105090', 'tt7440160', 'tt5260020', 'tt0049557', 'tt21426372', 'tt9121506', 'tt0068252', 'tt29134207', 'tt28523668', 'tt12985888', 'tt14309538', 'tt7694184', 'tt0830914', 'tt3605208', 'tt10987040', 'tt7512772', 'tt8461390', 'tt39902017', 'tt8870570', 'tt6886734', 'tt20752324', 'tt8386652', 'tt12221628', 'tt8221948', 'tt14986234', 'tt4801224', 'tt28106712', 'tt13561432', 'tt11388224', 'tt0167971', 'tt1508694', 'tt8037658', 'tt4818952', 'tt18926886', 'tt2461740', 'tt0282885', 'tt28655610', 'tt1822212', 'tt10456236', 'tt2816152', 'tt5463152', 'tt6590290', 'tt2286815', 'tt38973043', 'tt11349332', 'tt9454842', 'tt2548096', 'tt14629636', 'tt10280188', 'tt8042526', 'tt28676878', 'tt4498818', 'tt1059135

In [22]:
iran_basics = df_basics[df_basics['tconst'].isin(iran_titles_v3)]
iran_basics['titleType'].value_counts()

titleType
movie           986
short           463
tvSeries        233
tvMovie          75
video            47
tvMiniSeries     36
videoGame         8
tvSpecial         4
tvShort           2
Name: count, dtype: int64

In [23]:
selected_types = ['movie', 'tvSeries', 'tvMovie', 'tvMiniSeries']
iranian_tconst = df_basics[df_basics['tconst'].isin(iran_titles_v3) & df_basics['titleType'].isin(selected_types)]
iran_tconst_set = set(iranian_tconst['tconst'])
print("final Irainian set of films: ", len(iran_tconst_set))

final Irainian set of films:  1330


In [24]:
chunks = []
for chunk in pd.read_csv('../data/raw/title.principals.tsv.gz', sep='\t', chunksize=1_000_000, low_memory=False):
    filtered = chunk[chunk['tconst'].isin(iran_tconst_set)]
    chunks.append(filtered)

df_principals_iran = pd.concat(chunks, ignore_index=True)

df_merged = iranian_tconst.merge(df_principals_iran, on='tconst', how='inner')
print(df_merged.shape)
df_merged.head()

(17107, 14)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,nconst,category,job,characters
0,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",1,nm0798270,actor,\N,"[""Fernando Torres (Huracán Ramírez)""]"
1,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",2,nm0739131,actress,\N,"[""Margarita""]"
2,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",3,nm0327643,actress,\N,"[""Laura""]"
3,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",4,nm0273508,actor,\N,"[""El pichi""]"
4,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",5,nm0414109,actor,\N,"[""Señor Torres (Tonina Jackson)""]"


In [15]:
# saved df_merged on hard disk to use later 
import os
os.makedirs('../data/processed', exist_ok=True)
df_merged.to_parquet('../data/processed/iran_films_principals.parquet')
print("saved: ", df_merged.shape)

saved:  (92518, 14)


In [47]:
df_akas[df_akas['titleId'] == 'tt0044728']

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
417152,tt0044728,1,Huracán Ramírez,\N,\N,original,\N,1
417153,tt0044728,2,Huracán Ramírez,BR,\N,imdbDisplay,\N,0
417154,tt0044728,3,Huracán Ramírez,MX,\N,imdbDisplay,\N,0
417155,tt0044728,4,Qahremane shekastnapazir,IR,fa,imdbDisplay,\N,0
